# 🍥 AI Dubbing Studio — Kokoro-82M TTS + RVC (RMVPE)
### Optimized for Google Colab Free T4 GPU

**Pipeline:** Kokoro-82M TTS → RVC Voice Conversion (RMVPE) → FFmpeg Stitching → Gradio UI

| Component | Library | Status |
|---|---|---|
| Base TTS | `kokoro` | ✅ Pure Python, no compile |
| Voice Conversion | `infer-rvc-python` | ✅ **fairseq-FREE** fork, no compile |
| Pitch Extraction | RMVPE + ONNX GPU | ✅ T4 GPU accelerated |
| Audio Stitching | FFmpeg + pydub | ✅ Lossless |

> 👉 **Required:** `Runtime → Change runtime type → T4 GPU` before running any cell.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Verify T4 GPU
# ══════════════════════════════════════════════════════════════════════════════
!nvidia-smi
import torch
assert torch.cuda.is_available(), '❌ GPU NOT FOUND! Go to Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Install ALL Dependencies (Zero-Hang, Colab T4 Tested)
# ══════════════════════════════════════════════════════════════════════════════
#
# KEY CHANGE: We use `infer-rvc-python` instead of `rvc-python`.
# infer-rvc-python is a maintained fairseq-FREE fork:
#   ✅ No C++ compilation  ✅ No fairseq  ✅ Installs in < 60 seconds
#

# ── 2a. System packages ──────────────────────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng

# ── 2b. Clone / update project from GitHub ──────────────────────────────────
import os
if not os.path.exists('/content/narutogen'):
    !git clone https://github.com/adarshraj990/-narutogen.git /content/narutogen
else:
    !cd /content/narutogen && git pull --rebase origin main
%cd /content/narutogen

# ── 2c. Pin NumPy < 2.0 FIRST (critical ABI guard for all RVC internals) ────
!pip install -q "numpy<2.0.0"

# ── 2d. pyworld (pre-built binary — skips ALL C++ Cython compilation) ───────
!pip install -q pyworld-prebuilt

# ── 2e. infer-rvc-python — THE key package (fairseq-FREE, binary wheels) ────
!pip install -q --prefer-binary infer-rvc-python

# ── 2f. GPU-accelerated FAISS + ONNX Runtime ────────────────────────────────
!pip install -q faiss-cpu onnxruntime-gpu

# ── 2g. All remaining runtime packages ──────────────────────────────────────
!pip install -q \
    hydra-core \
    omegaconf \
    "antlr4-python3-runtime==4.9.3" \
    kokoro \
    soundfile \
    pydub \
    pysrt \
    librosa \
    scipy \
    "gradio>=4.44.1" \
    huggingface_hub \
    requests \
    psutil

# ── 2h. Verify RVC import (LOUD failure if broken) ──────────────────────────
import subprocess, sys
verify = subprocess.run(
    [sys.executable, '-c',
     'from infer_rvc_python.infer import RVCInference; '
     'print("✅ infer_rvc_python.RVCInference:", RVCInference)'],
    capture_output=True, text=True
)
if verify.returncode != 0:
    print('STDOUT:', verify.stdout)
    print('STDERR:', verify.stderr)
    raise RuntimeError('🚨 infer_rvc_python failed to import! Fix errors above before proceeding.')
print(verify.stdout.strip())
print('\n✅ All dependencies installed and verified. Proceed to Cell 3.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Launch AI Dubbing Studio
# ══════════════════════════════════════════════════════════════════════════════
#
# Generates a public https://xxxx.gradio.live share link.
# Saves output audio to /content/narutogen/outputs/
#
# ⚠️  If you see an import error for infer_rvc_python:
#      → Runtime → Restart runtime → skip Cell 2 → run Cell 3 directly
#
!python app.py